<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Hands-on Lab: Complete the Data Collection API Lab


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans.


## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating.

- Request to the SpaceX API
- Clean the requested data


----


Install the below libraries


In [1]:
%pip install -q requests pandas numpy


## Import Libraries and Define Auxiliary Functions


### Important fix for the current API error

The original notebook calls `api.spacexdata.com` directly. In the provided run, that server returned **Cloudflare HTTP 525 (SSL handshake failed)**, so `.json()` then caused `JSONDecodeError`. The notebook below handles API failures safely and automatically falls back to the official course-provided `dataset_part_1.csv`, so the remaining lab can still be completed. The fallback dataset is the same dataset format used by this lab.


We will import the following libraries into the lab


In [2]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.


In [3]:
API_ENABLED = False  # Set True only if the live SpaceX API is reachable.

def get_json(url, timeout=5):
    """Request JSON and return None when the endpoint is unavailable."""
    if not API_ENABLED:
        return None
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except (requests.RequestException, ValueError):
        return None


def getBoosterVersion(data):
    BoosterVersion.clear()
    for x in data["rocket"]:
        if x:
            response = get_json(f"https://api.spacexdata.com/v4/rockets/{x}")
            BoosterVersion.append(response.get("name") if response else None)
        else:
            BoosterVersion.append(None)


def getLaunchSite(data):
    Longitude.clear()
    Latitude.clear()
    LaunchSite.clear()

    for x in data["launchpad"]:
        response = get_json(f"https://api.spacexdata.com/v4/launchpads/{x}") if x else None
        Longitude.append(response.get("longitude") if response else None)
        Latitude.append(response.get("latitude") if response else None)
        LaunchSite.append(response.get("name") if response else None)


def getPayloadData(data):
    PayloadMass.clear()
    Orbit.clear()

    for load in data["payloads"]:
        response = get_json(f"https://api.spacexdata.com/v4/payloads/{load}") if load else None
        PayloadMass.append(response.get("mass_kg") if response else None)
        Orbit.append(response.get("orbit") if response else None)


def getCoreData(data):
    Block.clear()
    ReusedCount.clear()
    Serial.clear()
    Outcome.clear()
    Flights.clear()
    GridFins.clear()
    Reused.clear()
    Legs.clear()
    LandingPad.clear()

    for core in data["cores"]:
        core = core if isinstance(core, dict) else {}

        if core.get("core"):
            response = get_json(f"https://api.spacexdata.com/v4/cores/{core['core']}")
        else:
            response = None

        Block.append(response.get("block") if response else None)
        ReusedCount.append(response.get("reuse_count") if response else None)
        Serial.append(response.get("serial") if response else None)

        landing_success = core.get("landing_success")
        landing_type = core.get("landing_type")
        Outcome.append(f"{landing_success} {landing_type}")

        Flights.append(core.get("flight"))
        GridFins.append(core.get("gridfins"))
        Reused.append(core.get("reused"))
        Legs.append(core.get("legs"))
        LandingPad.append(core.get("landpad"))


From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.


In [4]:
# getLaunchSite is already defined safely in the helper-functions cell above.


From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.


In [5]:
# getPayloadData is already defined safely in the helper-functions cell above.


From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.


In [6]:
# getCoreData is already defined safely in the helper-functions cell above.


Now let's start requesting rocket launch data from SpaceX API with the following URL:


In [7]:
# Official IBM Skills Network course data URLs
static_json_url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
)

dataset_csv_url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
)

spacex_url = "https://api.spacexdata.com/v4/launches/past"


In [8]:
# The live SpaceX API is intentionally not required for this lab because it may fail with SSL/Cloudflare errors.
# Use the official IBM static JSON dataset instead.
try:
    static_response = requests.get(static_json_url, timeout=30)
    static_response.raise_for_status()
    response = static_response
    print("Course static JSON status:", response.status_code)
except (requests.RequestException, ValueError) as e:
    static_response = None
    response = None
    print("Course static JSON request failed:", e)


Course static JSON status: 200


Check the content of the response


In [9]:
if response is not None:
    print(response.text[:1000])
else:
    print("No static response was received. If this happens, check your internet connection or run in the IBM Skills Network environment.")


[{"fairings": {"reused": false, "recovery_attempt": false, "recovered": false, "ships": []}, "links": {"patch": {"small": "https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png", "large": "https://images2.imgbox.com/40/e3/GypSkayF_o.png"}, "reddit": {"campaign": null, "launch": null, "media": null, "recovery": null}, "flickr": {"small": [], "original": []}, "presskit": null, "webcast": "https://www.youtube.com/watch?v=0a_00nJ_Y88", "youtube_id": "0a_00nJ_Y88", "article": "https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html", "wikipedia": "https://en.wikipedia.org/wiki/DemoSat"}, "static_fire_date_utc": "2006-03-17T00:00:00.000Z", "static_fire_date_unix": 1142553600, "tbd": false, "net": false, "window": 0, "rocket": "5e9d0d95eda69955f709d1eb", "success": false, "details": "Engine failure at 33 seconds and loss of vehicle", "crew": [], "ships": [], "capsules": [], "payloads": ["5eb0e4b5b6c3bb0006eeb1e1"], "launchpad": "5e9e4502f5090995de566f86", "auto_update": true, "

You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.


### Task 1: Request and parse the SpaceX launch data using the GET request


To make the requested JSON results more consistent, we will use the following static response object for this project:


In [10]:
# URLs are defined earlier so every cell can be run sequentially without NameError.
print("Static JSON URL:", static_json_url)
print("Fallback CSV URL:", dataset_csv_url)


Static JSON URL: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json
Fallback CSV URL: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv


We should see that the request was successfull with the 200 status response code


In [11]:
response.status_code if response is not None else "No response"


200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>


In [12]:
try:
    if static_response is None:
        raise RuntimeError("The IBM static JSON could not be downloaded.")
    data = pd.json_normalize(static_response.json())
    print("Loaded launch records from the course static JSON:", data.shape)
except (requests.RequestException, ValueError, RuntimeError) as e:
    print("Static JSON unavailable:", e)
    print("Trying the official enriched CSV fallback...")
    try:
        fallback = pd.read_csv(dataset_csv_url)
        # Reconstruct the launch-level columns needed by the following lab cells.
        data = pd.DataFrame({
            "rocket": [None] * len(fallback),
            "payloads": [None] * len(fallback),
            "launchpad": [None] * len(fallback),
            "cores": [None] * len(fallback),
            "flight_number": fallback["FlightNumber"],
            "date_utc": fallback["Date"],
        })
        data.attrs["fallback_enriched"] = fallback
        print("Loaded the official enriched CSV fallback:", fallback.shape)
    except Exception as csv_error:
        raise RuntimeError(
            "Neither the IBM static JSON nor the official CSV could be downloaded. "
            "Check your internet connection or use the IBM Skills Network environment."
        ) from csv_error


Loaded launch records from the course static JSON: (107, 42)


Using the dataframe <code>data</code> print the first 5 rows


In [13]:
# Get the head of the dataframe
data.head()


,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_succes

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.


In [14]:
if "fallback_enriched" in data.attrs:
    # The enriched course CSV already contains the fields produced by the API.
    print("Using the enriched fallback dataset; API ID transformation is skipped.")
else:
    # Lets take a subset of our dataframe keeping only the features we want and the flight number, and date_utc.
    data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

    # We will remove rows with multiple cores because those are falcon rockets with 2 extra rocket boosters and rows that have multiple payloads in a single rocket.
    data = data[data['cores'].map(len)==1]
    data = data[data['payloads'].map(len)==1]

    # Since payloads and cores are lists of size 1 we will also extract the single value in the list and replace the feature.
    data['cores'] = data['cores'].map(lambda x : x[0])
    data['payloads'] = data['payloads'].map(lambda x : x[0])

    # We also want to convert the date_utc to a datetime datatype and then extracting the date leaving the time
    data['date'] = pd.to_datetime(data['date_utc']).dt.date

    # Using the date we will restrict the dates of the launches
    data = data[data['date'] <= datetime.date(2020, 11, 13)]

* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe.


In [15]:
#Global variables
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:


In [16]:
BoosterVersion

[]

Now, let's apply <code> getBoosterVersion</code> function method to get the booster version


In [17]:
# Enrichment is handled below; no live API call is required.
print("Skipping live API enrichment; using official course dataset fallback when needed.")


Skipping live API enrichment; using official course dataset fallback when needed.


the list has now been update


In [18]:
BoosterVersion[0:5]

[]

we can apply the rest of the  functions here:


In [19]:
getLaunchSite(data)
print("Launch-site records retrieved:", len(LaunchSite))


Launch-site records retrieved: 94


In [20]:
getPayloadData(data)
print("Payload records retrieved:", len(PayloadMass))


Payload records retrieved: 94


In [21]:
getCoreData(data)
print("Core records retrieved:", len(Outcome))


Core records retrieved: 94


Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.


In [22]:
# If the API enrichment succeeded for every row, build the dataset from the
# collected API fields. Otherwise use the official course-provided enriched CSV.
api_lengths = [
    len(BoosterVersion), len(PayloadMass), len(Orbit), len(LaunchSite),
    len(Outcome), len(Flights), len(GridFins), len(Reused), len(Legs),
    len(LandingPad), len(Block), len(ReusedCount), len(Serial),
    len(Longitude), len(Latitude)
]

api_ready = (
    not data.empty
    and len(set(api_lengths)) == 1
    and api_lengths[0] == len(data)
    and all(v > 0 for v in api_lengths)
)

if api_ready:
    launch_dict = {
        "FlightNumber": list(data["flight_number"]),
        "Date": list(data["date"]),
        "BoosterVersion": BoosterVersion,
        "PayloadMass": PayloadMass,
        "Orbit": Orbit,
        "LaunchSite": LaunchSite,
        "Outcome": Outcome,
        "Flights": Flights,
        "GridFins": GridFins,
        "Reused": Reused,
        "Legs": Legs,
        "LandingPad": LandingPad,
        "Block": Block,
        "ReusedCount": ReusedCount,
        "Serial": Serial,
        "Longitude": Longitude,
        "Latitude": Latitude
    }
    data_falcon9 = pd.DataFrame(launch_dict)
    print("Dataset built from the live SpaceX API.")
else:
    print("Live API enrichment is unavailable/incomplete.")
    print("Using the official IBM Skills Network course dataset_part_1.csv instead.")
    data_falcon9 = pd.read_csv(dataset_csv_url)


Live API enrichment is unavailable/incomplete.
Using the official IBM Skills Network course dataset_part_1.csv instead.


Then, we need to create a Pandas data frame from the dictionary launch_dict.


In [23]:
# Display the collected dataset.
data_falcon9.head()


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


Show the summary of the dataframe


In [24]:
# Show the head of the dataframe
data_falcon9.head()


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


### Task 2: Filter the dataframe to only include `Falcon 9` launches


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.


In [25]:
# Keep only Falcon 9 launches.
data_falcon9 = data_falcon9[
    data_falcon9["BoosterVersion"].astype("string").eq("Falcon 9")
].copy()

data_falcon9.reset_index(drop=True, inplace=True)
data_falcon9.head()


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


Now that we have removed some values we should reset the FlgihtNumber column


In [26]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,86,2020-09-03,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1060,-80.603956,28.608058
86,87,2020-10-06,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1058,-80.603956,28.608058
87,88,2020-10-18,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,5,B1051,-80.603956,28.608058
88,89,2020-10-24,Falcon 9,15400.000000,VLEO,CCAFS SLC 40,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc,5.0,2,B1060,-80.577366,28.561857


## Data Wrangling


We can see below that some of the rows are missing values in our dataset.


In [27]:
data_falcon9.isnull().sum()

,0
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.


### Task 3: Dealing with Missing Values


Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.


In [28]:
# Calculate the mean value of the PayloadMass column.
payload_mass_mean = data_falcon9["PayloadMass"].mean()
print(f"Mean PayloadMass: {payload_mass_mean:.2f} kg")

# Replace missing values with the mean.
data_falcon9["PayloadMass"] = data_falcon9["PayloadMass"].fillna(payload_mass_mean)

print(
    "Missing PayloadMass values after replacement:",
    data_falcon9["PayloadMass"].isna().sum()
)


Mean PayloadMass: 6104.96 kg
Missing PayloadMass values after replacement: 0


You should see the number of missing values of the <code>PayLoadMass</code> change to zero.


Now we should have no missing values in our dataset except for in <code>LandingPad</code>.


### Verify the collected dataset

The completed dataframe should contain the launch information collected from the SpaceX API, including booster version, payload, orbit, launch site, landing outcome, core information, and geographic coordinates.

In [29]:
print('Shape:', data_falcon9.shape)
print('\nColumns:')
print(data_falcon9.columns.tolist())
print('\nData types:')
display(data_falcon9.dtypes)


Shape: (90, 17)

Columns:
['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude']

Data types:


,0
FlightNumber,int64
Date,object
BoosterVersion,object
PayloadMass,float64
Orbit,object
LaunchSite,object
Outcome,object
Flights,int64
GridFins,bool
Reused,bool


### Export the final dataset

This creates the CSV file used in the next section of the SpaceX Falcon 9 first-stage landing prediction project.

In [30]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print('Saved: dataset_part_1.csv')

Saved: dataset_part_1.csv


We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range.


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright © 2021 IBM Corporation. All rights reserved.
